## Logistic Regression Model for Classfying Benign and Malignent Lesions

In [66]:
pip install kagglehub

Note: you may need to restart the kernel to use updated packages.


In [67]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import os
import pandas as pd
import kagglehub

In [68]:
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
csv_path = os.path.join(path, "HAM10000_metadata.csv")
df = pd.read_csv(csv_path)

### **Preprocssing Data:**
**Encodings:** 

For Logistic Regression, we are splitting up the data into binary encodings based on whether the diagnosis is malignant. If the dx (diagnosis) is **mel, bxx, or akiec** then it belongs to class 1, else it is benign and class 0. 

**Cleaning:**

We choose to not include dx_type within the initial model because diagnois type can introduce some underlying bias based on how it was diagnosed.

The dataset include some missing ages, which were filled in as the median age within the dataset to ensure a complete dataset and this will also be scaled. We removed image_id as it was unnecessary to the Logistic Regression. Finally, there is images of the same lesion which have the same lesion_id. To prevent data leakage, we grouped together all data points with the same lesion_id such that it is not split among the train and test splits. We achieve this using sklearn's GroupShuffleSplit.

In [69]:
# Preprocessing
def preprocess(data):
    cleaned_df = data.copy()
    cleaned_df = cleaned_df.drop(columns=["image_id"])

    cleaned_df["age"] = cleaned_df["age"].fillna(cleaned_df["age"].median())

    malignant = ["mel", "bcc", "akiec"]
    cleaned_df["target"] = cleaned_df["dx"].isin(malignant).astype(int)

    X = cleaned_df.drop(columns=["dx", "target", "lesion_id", "dx_type"])
    y = cleaned_df["target"]

    X = pd.get_dummies(X, columns=["sex", "localization"], drop_first=True)

    return X, y, cleaned_df["lesion_id"]

In [70]:
# Logistic Regression: 
def logistic_regression(data, iter=1000, threshold=0.5, class_weight=None, reg=1):
    '''
        Trains Logistic Regression

        Args:
            data: the dataset
            iter = number of iterations to train the model one
            threshold = the probabilistic threshold to determine 1 or 0
            class_weight = how much the model penalizes misclassification of malignant lesions
            reg = regularization strength
        
        Returns:
            accuracy, precision, f1, and recall scores
    '''
    X, y, groups = preprocess(data)

    gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
    )

    train_idx, test_idx = next(gss.split(X, y, groups))

    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

    scaler = StandardScaler()


    X_train["age"] = scaler.fit_transform(X_train[["age"]])
    X_test["age"] = scaler.transform(X_test[["age"]])
    model = LogisticRegression(max_iter=iter, class_weight=class_weight, C=reg)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob > threshold).astype(int)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1:", f1_score(y_test, y_pred))  

In [71]:
logistic_regression(df, iter=1000)

Accuracy: 0.7959486166007905
Precision: 0.4375
Recall: 0.178117048346056
F1: 0.25316455696202533


In [72]:
logistic_regression(df, iter=1000, threshold=0.7)

Accuracy: 0.8068181818181818
Precision: 0.75
Recall: 0.007633587786259542
F1: 0.015113350125944584


In [73]:
logistic_regression(df, iter=1000, threshold=0.2)

Accuracy: 0.7099802371541502
Precision: 0.38170731707317074
Recall: 0.7964376590330788
F1: 0.516075845012366


In [74]:
logistic_regression(df, iter=1000, threshold=0.2, class_weight="balanced")

Accuracy: 0.4105731225296443
Precision: 0.2425997425997426
Recall: 0.9592875318066157
F1: 0.3872624550590652


In [75]:
logistic_regression(df, iter=1000, class_weight="balanced")

Accuracy: 0.7109683794466403
Precision: 0.38264058679706603
Recall: 0.7964376590330788
F1: 0.5169281585466556


In [76]:
logistic_regression(df, iter=1000, class_weight="balanced", reg=0.1)

Accuracy: 0.7060276679841897
Precision: 0.375
Recall: 0.7709923664122137
F1: 0.5045795170691091
